# 분산 분석(ANOVA)

## formula(수식) 작성 문법 정리

### ~ (물결)
- 종속변수(타겟, 정답)와 독립변수(피처, 입력) 구분
- target ~ a
    - target : 종속변수
    - a : 독립변수

### + (더하기)
- 여러 독립변수를 모델에 포함
- target ~ a + b + c
    - target : 종속변수
    - a, b, c : 여러 독립변수

### C(변수)
- 해당 변수를 범주형(Categorical)으로 명시
- 변수가 이미 문자열이면 자동 인식되지만, 숫자로 코딩된 범주형 변수는 반드시 C()로 감싸야 연속형으로 잘못 처리되지 않는다
- 컬럼명에 띄어쓰기나 특수문자가 있으면 formula 문법에서 에러가 난다
    - 먼저 df.rename(columns={'원래 컬럼명':'새 컬럼명'}) 컬럼명 수정한 뒤 모델을 만들어야 한다

In [1]:
import pandas as pd
df = pd.read_csv('fertilizer.csv')
df.head()

,비료,성장
0,A,10.5
1,A,11.3
2,A,10.8
3,A,9.6
4,A,11.1


### anova_lm() 출력 표 읽기
- df (자유도, degree of freedom) : 그룹 수 - 1(요인의 자유도), 전체 관측치 수 - 그룹 수 (잔차의 자유도)
- sum_sq (제곱합, ss) : 그룹별 또는 잔차의 제곱합 (sum of squares)
- mean_sq (평균제곱, ms) : sum_sq / df
- F (F-통계량) : 그룹 간 분산 / 그룹 내 분산 비율
- PR(>F) (p-value) : F-통계량에 대한 유의 확률                            

In [2]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

model = ols('성장 ~ C(비료)', df).fit()
anova_lm(model)

,df,sum_sq,mean_sq,F,PR(>F)
C(비료),3.0,43.21875,14.406250,89.126139,1.001838e-16
Residual,36.0,5.81900,0.161639,NaN,NaN


## 이원 분산 분석(Two-way ANOVA)
- 독립변수가 2개(예: 나무 종류, 비료 종류)일 때 각각의 영향과 두 변수의 조합(상호작용) 효과까지 분석
- 기본 가정은 일원 분산 분석과 동일하게 독립성, 정규성(Shapiro-Wilk), 등분산성(Levene)으로 한다

- 주효과 2개 + 상호작용 효과 1개 → 총 3가지 가설을 동시에 검정
    1. 주효과 - 요인 A (예: 학습 방법)
        - 귀무가설 : 학습 방법에 따라 성적 차이가 없다
        - 대립가설 : 학습 방법에 따라 성적 차이가 있다
    2. 주효과 - 요인 B (예: 학습 장소)
        - 귀무가설 : 학습 장소에 따라 성적 차이가 없다
        - 대립가설 : 학습 장소에 따라 성적 차이가 있다
    3. 상호작용 효과
        - 귀무가설 : A 요인과 B 요인 간에 상호작용이 없다
        - 상호작용 : A 요인과 B 요인 간에 상호작용이 있다

- statmodels의 ols와 anova_lm 사용

In [4]:
df = pd.read_csv('tree.csv')
df.sample(10)

,나무,비료,성장률
29,A,3,53.083063
63,C,1,48.037934
106,D,2,86.861859
4,A,1,47.658466
92,D,1,57.979469
60,C,1,55.208258
102,D,2,64.572855
5,A,1,47.658630
75,C,2,71.219025
2,A,1,56.476885


In [5]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# 성장률 ~ 나무 + 비료 + 나무:비료
model = ols('성장률 ~ 나무 + 비료 + 나무:비료', data=df).fit()
anova_table = sm.stats.anova_lm(model)
anova_table

,df,sum_sq,mean_sq,F,PR(>F)
나무,3.0,4783.353938,1594.451313,18.391274,9.016693e-10
비료,1.0,873.322002,873.322002,10.073374,1.942421e-03
나무:비료,3.0,394.801585,131.600528,1.517952,2.137666e-01
Residual,112.0,9709.960792,86.696078,NaN,NaN


In [6]:
model = ols('성장률 ~ C(나무) + C(비료) + C(나무):C(비료)', data=df).fit()
anova_table = sm.stats.anova_lm(model)
anova_table

,df,sum_sq,mean_sq,F,PR(>F)
C(나무),3.0,4783.353938,1594.451313,18.855528,6.600012e-10
C(비료),2.0,1127.924259,563.962129,6.669256,1.857612e-03
C(나무):C(비료),6.0,717.520672,119.586779,1.414199,2.157357e-01
Residual,108.0,9132.639448,84.561476,NaN,NaN


In [7]:
print(format(6.600012e-10, '.11f'))

0.00000000066


In [8]:
print(format(1.857612e-03, '.4f'))

0.0019


In [9]:
print(format(2.157357e-01, '.4f'))

0.2157


#### 상호작용 효과를 *로 간단히 표현하기
- '나무 + 비료 + 나무:비료' → '나무 * 비료'

In [10]:
model = ols('성장률 ~ C(나무) * C(비료)', data=df).fit()
anova_table = sm.stats.anova_lm(model)
anova_table

,df,sum_sq,mean_sq,F,PR(>F)
C(나무),3.0,4783.353938,1594.451313,18.855528,6.600012e-10
C(비료),2.0,1127.924259,563.962129,6.669256,1.857612e-03
C(나무):C(비료),6.0,717.520672,119.586779,1.414199,2.157357e-01
Residual,108.0,9132.639448,84.561476,NaN,NaN


- 나무 종류 → 성장률에 유의미한 차이가 있다 - p < 0.05
- 비료 종류 → 성장률에 유의미한 차이가 있다 - p < 0.05
- 나무와 비료 간의 상호작용은 성장률에 유의미한 영향을 주지 않는다 - p >= 0.05

- 주효과 2개는 유의미, 상호작용은 유의미하지 않은 패턴이 시험에 자주 출제

### 레빈 검정 (Levene)
- center 파라미터에 따라 mean(Levene 검정) 또는 median(Brown-Forsythe 검정)으로 등분산성을 확인할 수 있다
    - 문제에서 특별히 명시하지 않으면 기본값 사용

### anova_lm(model, typ=숫자)
- typ 파라미터로 제곱합 계산 방식 조정 가능
- typ=1 (기본값) → 문제에서 특별한 계산 방식을 요구하지 않으면 이것을 사용 / 모델이 단순한 경우(설계가 균형잡힌 경우) typ 값을 바꿔도 결과가 유사하게 나오는 경우가 많다
    - 순차적 제곱합 : 변수를 하나씩 추가하며 순서대로 영향을 계산
- typ=2
    - 불순차적 제곱합 : 각 변수가 독립적으로 미치는 영향을 계산
- typ=3
    - 제곱합 : 상호작용까지 고려한 상태에서 각 변수의 효과 계산